In [34]:
import os
# This will list the folders in your input. 
# Look for 'asl-signs' or a long string of numbers.
print(os.listdir('/kaggle/input/'))

['competitions']


In [40]:
import pandas as pd
import numpy as np
import os
import shutil

# 1. CLEANUP: Delete all old failed attempts and zips to free space
!rm -rf /kaggle/working/*

# 2. PATH SETUP
# Let's try to find the CSV again
base_input = '/kaggle/input/competitions/asl-signs'
csv_path = os.path.join(base_input, 'train.csv')

train_df = pd.read_csv(csv_path)
target_words = [
    'minemy', 'yourself', 'hesheit', 'weus', 
    'food', 'drink', 'have', 'hungry', 'sleep', 
    'water', 'home', 'store', 'go', 'wait', 
    'stay', 'now', 'hello', 'please', 'thankyou', 'yes', 'no'
]
filtered_df = train_df[train_df['sign'].isin(target_words)]

SAVE_DIR = '/kaggle/working/suraj_final_data'
os.makedirs(SAVE_DIR, exist_ok=True)

# 3. TEST ONE FILE (The 'Canary' test)
test_row = filtered_df.iloc[0]
test_path = os.path.join(base_input, test_row['path'])
print(f"🔍 Testing path: {test_path}")

try:
    test_df = pd.read_parquet(test_path)
    print("✅ Parquet read successful!")
except Exception as e:
    print(f"❌ FAILED TO READ FIRST FILE: {e}")
    # If this fails, the script stops here so you don't waste time
    raise

# 4. FULL EXTRACTION
print(f"🚀 Extracting {len(filtered_df)} videos...")
for i, (idx, row) in enumerate(filtered_df.iterrows()):
    full_pq_path = os.path.join(base_input, row['path'])
    
    try:
        df = pd.read_parquet(full_pq_path)
        # Extract Hands
        lh = df[df['type'] == 'left_hand'][['x', 'y', 'z']].fillna(0).values
        rh = df[df['type'] == 'right_hand'][['x', 'y', 'z']].fillna(0).values
        
        # Reshape to (Frames, 63)
        num_frames = df['frame'].nunique()
        features = np.hstack([lh.reshape(num_frames, 63), rh.reshape(num_frames, 63)])
        
        # Save
        word_dir = os.path.join(SAVE_DIR, row['sign'])
        if not os.path.exists(word_dir): os.makedirs(word_dir)
        np.save(os.path.join(word_dir, f"{row['sequence_id']}.npy"), features.astype('float32'))
        
        if i % 1000 == 0: print(f"Done: {i}/{len(filtered_df)}")
    except Exception:
        continue

# 5. ZIP VIA SYSTEM (Faster and more reliable)
print("📦 Zipping...")
!zip -r -q suraj_final_v2.zip /kaggle/working/suraj_final_data
print(f"🎯 DONE. Zip Size: {os.path.getsize('suraj_final_v2.zip')/1e6:.2f} MB")



🔍 Testing path: /kaggle/input/competitions/asl-signs/train_landmark_files/28656/1000106739.parquet
✅ Parquet read successful!
🚀 Extracting 7902 videos...
Done: 0/7902
Done: 1000/7902
Done: 2000/7902
Done: 3000/7902
Done: 4000/7902
Done: 5000/7902
Done: 6000/7902
Done: 7000/7902
📦 Zipping...
🎯 DONE. Zip Size: 44.68 MB
❌ Error: Zip file was not created. Check if disk space is full (20GB limit).


In [42]:
zip_path = '/kaggle/working/suraj_final_v2.zip'
if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"📊 Final Zip Size: {size_mb:.2f} MB")
    
    # Method 1: IPython FileLink
    display(FileLink(zip_path))
    
    # Method 2: Manual HTML Link (Alternative)
    html = f'<a href="{zip_path}" download="{zip_path}"><b>Click Here to Download Your Zip File</b></a>'
    display(HTML(html))
else:
    print("❌ Error: Zip file was not created. Check if disk space is full (20GB limit).")

📊 Final Zip Size: 42.61 MB


/kaggle/working/suraj_final_v2.zip

In [43]:
from IPython.display import FileLink, HTML
import os

zip_file = 'suraj_final_v2.zip'

if os.path.exists(zip_file):
    print(f"✅ Zip found! Size: {os.path.getsize(zip_file)/1e6:.2f} MB")
    
    # Method 1: The Standard Kaggle Link
    print("Method 1 (Standard):")
    display(FileLink(zip_file))
    
    # Method 2: The HTML 'Anchor' Link (Harder to block)
    print("\nMethod 2 (HTML):")
    html_link = f'<a href="{zip_file}" download="{zip_file}" style="color: #00ff00; font-size: 20px;"><b>🚀 CLICK HERE TO DOWNLOAD ZIP</b></a>'
    display(HTML(html_link))
else:
    print("❌ Zip file not found in /kaggle/working/. Did the zip command fail?")

✅ Zip found! Size: 44.68 MB
Method 1 (Standard):


/kaggle/working/suraj_final_v2.zip


Method 2 (HTML):
